This template shows how to use Matplotlib to create charts, since charting is a common feature in research.

## Preparation

Import matplotlib and get some historical data to plot.

In [ ]:
# Import the matplotlib library for plots and settings.
import matplotlib.pyplot as plt
import mplfinance       # for candlestick
import numpy as np
from pandas.plotting import register_matplotlib_converters
register_matplotlib_converters()

In [ ]:
# Get historical data for a bank sector ETF and some banking companies over 2021.
qb = QuantBook()
tickers = ["XLF",   # Financial Select Sector SPDR Fund
           "COF",   # Capital One Financial Corporation
           "GS",    # Goldman Sachs Group, Inc.
           "JPM",   # J P Morgan Chase & Co
           "WFC"]   # Wells Fargo & Company
symbols = [qb.add_equity(ticker).symbol for ticker in tickers]
history = qb.history(symbols, datetime(2021, 1, 1), datetime(2022, 1, 1))

## Candlestick Chart

In [ ]:
# Select a symbol and grab its OHLCV data.
symbol = symbols[0]
data = history.loc[symbol]
data.columns = ['Close', 'High', 'Low', 'Open', 'Volume']

# Plot the candlesticks.
mplfinance.plot(data,
                type='candle',
                style='charles',
                title=f'{symbol.value} OHLC',
                ylabel='Price ($)',
                figratio=(15, 10))

## Line Plot

In [ ]:
# Plot the close price of the first symbol.
data = history.loc[symbol]['close']
data.plot(title=f"{symbol} Close Price", figsize=(15, 10))

## Scatter Plot

In [ ]:
# Select 2 stocks and compute their daily returns.
symbol1 = symbols[1]
symbol2 = symbols[2]
close_price1 = history.loc[symbol1]['close']
close_price2 = history.loc[symbol2]['close']
daily_returns1 = close_price1.pct_change().dropna()
daily_returns2 = close_price2.pct_change().dropna()

# Fit an OLS line to the returns.
m, b = np.polyfit(daily_returns1, daily_returns2, deg=1)
x = np.linspace(daily_returns1.min(), daily_returns1.max())
y = m*x + b

# Plot regression line and scatter dots.
plt.plot(x, y, color='red')
plt.scatter(daily_returns1, daily_returns2)
plt.title(f'{symbol1} vs {symbol2} daily returns Scatter Plot')
plt.xlabel(symbol1.value)
plt.ylabel(symbol2.value)

## Histogram

In [ ]:
# Daily returns of the first symbol.
close_prices = history.loc[symbol]['close']
daily_returns = close_prices.pct_change().dropna()

# Normal distribution PDF using the empirical mean and std.
mean = daily_returns.mean()
std = daily_returns.std()
x = np.linspace(-3*std, 3*std, 1000)
pdf = 1/(std * np.sqrt(2*np.pi)) * np.exp(-(x-mean)**2 / (2*std**2))

# Plot the PDF over the histogram.
plt.plot(x, pdf, label="Normal Distribution")
plt.hist(daily_returns, bins=20)
plt.title(f'{symbol} Return Distribution')
plt.xlabel('Daily Return')
plt.ylabel('Count')

## Bar Chart

In [ ]:
# Mean daily % return per ticker.
close_prices = history['close'].unstack(level=0)
daily_returns = close_prices.pct_change() * 100
avg_daily_returns = daily_returns.mean()

plt.figure(figsize=(15, 10))
plt.bar([s.value for s in avg_daily_returns.index], avg_daily_returns)
plt.title('Banking Stocks Average Daily % Returns')
plt.xlabel('Tickers')
plt.ylabel('%')

## Heat Map

In [ ]:
# Correlation matrix of daily returns.
close_prices = history['close'].unstack(level=0)
daily_returns = close_prices.pct_change()
corr_matrix = daily_returns.corr()

# Plot the heat map with ticker labels.
plt.imshow(corr_matrix, cmap='hot', interpolation='nearest')
plt.title('Banking Stocks and Bank Sector ETF Correlation Heat Map')
plt.xticks(np.arange(len(tickers)), labels=tickers)
plt.yticks(np.arange(len(tickers)), labels=tickers)
plt.colorbar()

## Pie Chart

In [ ]:
# Inverse-variance portfolio weights.
close_prices = history['close'].unstack(level=0)
daily_returns = close_prices.pct_change()
inverse_variance = 1 / daily_returns.var()

plt.pie(inverse_variance, labels=inverse_variance.index, autopct='%1.1f%%')
plt.title('Banking Stocks and Bank Sector ETF Allocation')

## 3D Chart

In [ ]:
# Pick three symbols and grab their close-price series.
x, y, z = symbols[:3]
x_hist = history.loc[x].close
y_hist = history.loc[y].close
z_hist = history.loc[z].close

# Build the 3D scatter.
fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(projection='3d')
ax.scatter(x_hist, y_hist, z_hist)
ax.set_xlabel(f"{x} Price")
ax.set_ylabel(f"{y} Price")
ax.set_zlabel(f"{z} Price")

# Zoom out a bit so the z-axis label isn't clipped.
ax.set_box_aspect(None, zoom=0.85)
plt.show()